# 01 — Feature Engineering
Road-level traffic volume prediction.

Reproduces `src/feature_engineering.py`: inspection → cleaning → features.
Source data is **never modified**; outputs go to `data/processed/`.

In [1]:
import os, sys
PROJ = os.getcwd()
while not os.path.exists(os.path.join(PROJ, 'src', 'feature_engineering.py')):
    PROJ = os.path.dirname(PROJ)
os.chdir(PROJ)
sys.path.insert(0, os.path.join(PROJ, 'src'))
import pandas as pd
pd.set_option('display.width', 160)
from feature_engineering import load_raw, clean, add_features, chronological_split_dates, FEATURES

## 1. Load raw ML dataset (read-only)

In [2]:
df = load_raw()
print(f'{len(df):,} rows x {df.shape[1]} cols')
df.head()

8,506,000 rows x 22 cols


,road_id,date,hour,highway,highway_code,road_length_m,lane_count,speed_limit_kmh,is_oneway,is_bridge,...,node_degree,connected_road_count,intersection_density,day_of_week,month,is_weekend,is_peak_hour,morning_peak,evening_peak,traffic_volume
0,osm_road_000000,2024-01-01,0,trunk,10,179.015427,2.0,80.0,1,0,...,3.0,2.0,2.211645,0,1,0,0,0,0,8
1,osm_road_000001,2024-01-01,0,trunk_link,11,421.320526,1.0,50.0,1,0,...,3.0,3.0,2.211645,0,1,0,0,0,0,228
2,osm_road_000002,2024-01-01,0,trunk,10,397.889130,4.0,80.0,1,0,...,3.0,1.0,2.211645,0,1,0,0,0,0,185
3,osm_road_000003,2024-01-01,0,trunk,10,272.795959,4.0,80.0,1,1,...,3.0,1.0,2.211645,0,1,0,0,0,0,320
4,osm_road_000004,2024-01-01,0,trunk,10,1350.156494,4.0,80.0,1,0,...,3.0,3.0,2.211645,0,1,0,0,0,0,0


## 2. Data-quality checks

In [3]:
mv = df.isna().sum()
print('missing values per column:\n', mv[mv > 0] if mv.any() else 'none')
print('duplicate keys:', df.duplicated(['road_id','date','hour']).sum())
t = df['traffic_volume']
print('\ntarget stats: min=%d max=%d mean=%.1f median=%.1f std=%.1f skew=%.2f zeros=%.1f%%'
      % (t.min(), t.max(), t.mean(), t.median(), t.std(), t.skew(), (t==0).mean()*100))

missing values per column:
 none


duplicate keys: 0



target stats: min=0 max=56941 mean=1306.4 median=145.0 std=2995.3 skew=5.20 zeros=36.8%


## 3. Cleaning decisions (see reports/leakage_audit.md)
- drop `intersection_density` — constant everywhere
- drop `highway_code` — arbitrary ordinal duplicate of `highway`
- `road_id` kept as identifier, excluded from model features
- `is_weekend` uses Fri/Sat convention (matches demand dip) — kept as-is

In [4]:
dfc = clean(df)
print('after clean:', dfc.shape)

after clean: (8506000, 20)


## 4. Engineered features

In [5]:
dfe = add_features(dfc.head(100_000).copy())
dfe[[c for c in FEATURES if c.startswith(('hour_','dow_','road_length_log','lane_speed'))]].head()

,hour_sin,hour_cos,dow_sin,dow_cos,road_length_log,lane_speed_product
0,0.0,1.0,0.0,1.0,5.193043,160.0
1,0.0,1.0,0.0,1.0,6.045764,50.0
2,0.0,1.0,0.0,1.0,5.988684,320.0
3,0.0,1.0,0.0,1.0,5.612383,320.0
4,0.0,1.0,0.0,1.0,7.208716,320.0


## 5. Chronological splits (determined from actual date range)

In [6]:
for k, (a, b) in chronological_split_dates().items():
    m = (df['date'] >= a) & (df['date'] <= b)
    print(f'{k:11s} {a} .. {b}: {m.sum():,} rows')

train       2024-01-01 .. 2024-04-30: 5,146,130 rows
validation  2024-05-01 .. 2024-06-15: 1,956,380 rows
test        2024-06-16 .. 2024-07-18: 1,403,490 rows


## Regenerate full cleaned dataset
`python src/feature_engineering.py` writes `data/processed/traffic_ml_clean.csv` (+ `.parquet` cache + `feature_metadata.json`).